# Parte 2.1 — Big Data Science
Ler CSV do Spotify, pré-processar e salvar em Parquet.

In [2]:
import os
import polars as pl

diretorio_atual = os.getcwd()

caminho_csv = os.path.join(diretorio_atual, "Spotify_Music (1).csv")
caminho_parquet = os.path.join(diretorio_atual, "spotify_music.parquet")

if not os.path.exists(caminho_csv):
    caminho_csv = os.path.join(diretorio_atual, "..", "Spotify_Music (1).csv")

if not os.path.exists(caminho_csv):
    raise FileNotFoundError(f"CSV não encontrado em: {diretorio_atual}")

print(f"📂 CSV: {caminho_csv}")
print(f"📦 Parquet: {caminho_parquet}")

📂 CSV: c:\Users\rafael.chousa\Documents\CursoAnalistaDadosSenac2026.2\projeto_final\parte2\Spotify_Music (1).csv
📦 Parquet: c:\Users\rafael.chousa\Documents\CursoAnalistaDadosSenac2026.2\projeto_final\parte2\spotify_music.parquet


In [3]:
lazy_df = pl.scan_csv(caminho_csv, infer_schema_length=10000)

In [4]:
lazy_df = (
    lazy_df
    # .drop_nulls(subset=["energy", "loudness", "popularity", "track_id"])  # <- remova se quiser 100%
    # .unique(subset=["track_id"])                                          # <- REMOVA ISSO
    .with_columns([
        pl.col("explicit").cast(pl.Boolean),
        pl.col("popularity").cast(pl.Int32),
        pl.col("duration_ms").cast(pl.Int64),
        pl.col("key").cast(pl.Int32),
        pl.col("mode").cast(pl.Int32),
        pl.col("time_signature").cast(pl.Int32),
        pl.col("danceability").cast(pl.Float64),
        pl.col("energy").cast(pl.Float64),
        pl.col("loudness").cast(pl.Float64),
        pl.col("tempo").cast(pl.Float64),
        pl.col("valence").cast(pl.Float64),
    ])
    .with_columns([
        (pl.col("duration_ms") / 60000).round(2).alias("duration_min"),
        pl.col("artists").str.split(";").list.first().alias("main_artist"),
    ])
)

In [5]:
lazy_df.sink_parquet(caminho_parquet, compression="snappy")
print(f"✅ Parquet gerado: {caminho_parquet}")

✅ Parquet gerado: c:\Users\rafael.chousa\Documents\CursoAnalistaDadosSenac2026.2\projeto_final\parte2\spotify_music.parquet


In [6]:
df_check = pl.read_parquet(caminho_parquet)
print(f"🔁 {df_check.shape[0]} linhas x {df_check.shape[1]} colunas")
df_check.head(3)

🔁 114000 linhas x 23 colunas


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration_min,main_artist
i64,str,str,str,str,i32,i64,bool,f64,f64,i32,f64,i32,f64,f64,f64,f64,f64,f64,i32,str,f64,str
0,"""5SuOikwiRyPMVoIQDJUgSV""","""Gen Hoshino""","""Comedy""","""Comedy""",73,230666,false,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,"""acoustic""",3.84,"""Gen Hoshino"""
1,"""4qPNDBW1i3p13qLCt0Ki3A""","""Ben Woodward""","""Ghost (Acoustic)""","""Ghost - Acoustic""",55,149610,false,0.42,0.166,1,-17.235,1,0.0763,0.924,0.000006,0.101,0.267,77.489,4,"""acoustic""",2.49,"""Ben Woodward"""
2,"""1iJBSr7s7jYXzM8EGcbK5b""","""Ingrid Michaelson;ZAYN""","""To Begin Again""","""To Begin Again""",57,210826,false,0.438,0.359,0,-9.734,1,0.0557,0.21,0.0,0.117,0.12,76.332,4,"""acoustic""",3.51,"""Ingrid Michaelson"""
